# 🩺 Smart Diabetes Monitoring and Prediction System
## Notebook 4 — Feature Engineering and Selection

Expands 21 features to 45, trains models on engineered and selected features (Phases 2 & 3). Saves all phase results for Notebook 5.

### Setup

In [11]:
import os, sys, time, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score, log_loss
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy.stats import pointbiserialr

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("✓ imbalanced-learn available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("✗ imbalanced-learn not installed")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False

try:
    import catboost as cb
    CAT_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CAT_AVAILABLE = False

print("\n✅ All libraries loaded.")

✓ imbalanced-learn available
✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

✅ All libraries loaded.


In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ────────────────────────
DATA_PATH   = r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Dataset\diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_ROOT = Path(r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Outputs_Folder")

DIRS = {
    "models"       : OUTPUT_ROOT / "models",
    "scalers"      : OUTPUT_ROOT / "scalers",
    "plots_eda"    : OUTPUT_ROOT / "plots" / "01_eda",
    "plots_imb"    : OUTPUT_ROOT / "plots" / "02_imbalance",
    "plots_pre"    : OUTPUT_ROOT / "plots" / "03_preprocessing",
    "plots_base"   : OUTPUT_ROOT / "plots" / "04_baseline_models",
    "plots_eng"    : OUTPUT_ROOT / "plots" / "05_feature_engineering",
    "plots_eng_m"  : OUTPUT_ROOT / "plots" / "06_engineered_models",
    "plots_fi"     : OUTPUT_ROOT / "plots" / "07_feature_importance",
    "plots_sel"    : OUTPUT_ROOT / "plots" / "08_selected_models",
    "plots_cmp"    : OUTPUT_ROOT / "plots" / "09_model_comparison",
    "plots_best"   : OUTPUT_ROOT / "plots" / "10_best_model",
    "reports"      : OUTPUT_ROOT / "reports",
    "features"     : OUTPUT_ROOT / "feature_lists",
    "thresholds"   : OUTPUT_ROOT / "thresholds",
    "pipeline"     : OUTPUT_ROOT / "pipeline",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Output root: {OUTPUT_ROOT.resolve()}")

✅ Output root: C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder


In [13]:
PALETTE   = ["#00B4D8","#0077B6","#48CAE4","#90E0EF",
             "#ADE8F4","#023E8A","#03045E","#CAF0F8"]
COLOR_POS = "#EF233C"
COLOR_NEG = "#00B4D8"
COLOR_ACC = "#2EC4B6"
COLOR_AUC = "#FF9F1C"
COLOR_ROC = "#0077B6"
COLOR_PR  = "#2EC4B6"
COLOR_F1  = "#FF9F1C"

sns.set_style("whitegrid")
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#F8FBFF",
    "axes.edgecolor"  : "#CCCCCC",
    "axes.titlesize"  : 14,
    "axes.labelsize"  : 12,
    "xtick.labelsize" : 10,
    "ytick.labelsize" : 10,
    "font.family"     : "DejaVu Sans",
})

SEED     = 42
CV_FOLDS = 5
TOP_N    = 15
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

all_results = {}

print(f"✅ SEED={SEED}  CV_FOLDS={CV_FOLDS}  TOP_N={TOP_N}")

✅ SEED=42  CV_FOLDS=5  TOP_N=15


In [14]:
def save_pkl(obj, path, label=""):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  [SAVED] {label} → {path}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def savefig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [PLOT]  {path.name}")

def section(title):
    print("\n" + "━"*70)
    print(f"  {title}")
    print("━"*70)

def print_classification_report(y_true, y_pred, model_name, phase=""):
    print(f"\n{'─'*60}")
    print(f"  📊 Classification Report — {model_name}")
    if phase:
        print(f"     Phase: {phase}")
    print(f"{'─'*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["No Diabetes (0)", "Diabetes (1)"],
        digits=4))
    print(f"{'─'*60}")

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name,
                   cv_obj=None, phase="", verbose=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = (model.predict_proba(X_te)[:, 1]
               if hasattr(model, "predict_proba") else None)
    metrics = dict(
        accuracy      = accuracy_score(y_te, y_pred),
        precision     = precision_score(y_te, y_pred, zero_division=0),
        recall        = recall_score(y_te, y_pred, zero_division=0),
        f1            = f1_score(y_te, y_pred, zero_division=0),
        balanced_acc  = balanced_accuracy_score(y_te, y_pred),
        mcc           = matthews_corrcoef(y_te, y_pred),
        roc_auc       = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
        avg_precision = average_precision_score(y_te, y_proba) if y_proba is not None else np.nan,
        log_loss_val  = log_loss(y_te, y_proba) if y_proba is not None else np.nan,
        train_time_s  = elapsed,
        cv_roc_auc    = np.nan,
    )
    if cv_obj is not None:
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_obj, scoring="roc_auc", n_jobs=-1)
        metrics["cv_roc_auc"] = scores.mean()
    if verbose:
        print(f"  {name:<35} Acc={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}  "
              f"Prec={metrics['precision']:.4f}  AUC={metrics['roc_auc']:.4f}  "
              f"t={elapsed:.1f}s")
        print_classification_report(y_te, y_pred, name, phase)
    return metrics

def plot_roc_pr(y_true, models_proba, title_prefix, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auc = roc_auc_score(y_true, proba)
        axes[0].plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
        prec, rec, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        axes[1].plot(rec, prec, lw=2, label=f"{name}  AP={ap:.3f}")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=.5)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{title_prefix} — ROC Curve")
    axes[0].legend(fontsize=7)
    baseline_ap = y_true.mean()
    axes[1].axhline(baseline_ap, color="gray", linestyle="--",
                    label=f"Baseline AP={baseline_ap:.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{title_prefix} — Precision-Recall Curve")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, save_dir / f"{title_prefix.replace(' ','_')}_roc_pr.png")
    plt.show()

def results_to_df(results_dict):
    rows = []
    for phase, models in results_dict.items():
        for mname, m in models.items():
            row = {"Phase": phase, "Model": mname}
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)

def get_models():
    m = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Extra Trees"        : ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Gradient Boosting"  : GradientBoostingClassifier(
            n_estimators=200, random_state=SEED),
        "AdaBoost"           : AdaBoostClassifier(
            n_estimators=100, random_state=SEED),
    }
    if XGB_AVAILABLE:
        m["XGBoost"] = xgb.XGBClassifier(
            n_estimators=200, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
            scale_pos_weight=6)
    if LGB_AVAILABLE:
        m["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, n_jobs=-1,
            class_weight="balanced", verbose=-1,
            scale_pos_weight=6,
            min_child_samples=5,
            min_split_gain=0.0)
    if CAT_AVAILABLE:
        m["CatBoost"] = cb.CatBoostClassifier(
            iterations=200, random_state=SEED, verbose=0,
            auto_class_weights="Balanced")
    return m

print("✅ All helper functions defined.")

✅ All helper functions defined.


In [15]:
def engineer_features(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["CompositeRiskScore"]  = (d["HighBP"] + d["HighChol"] +
                                 d["Smoker"] + d["Stroke"] +
                                 d["HeartDiseaseorAttack"] + d["DiffWalk"])
    d["BMI_Underweight"]     = (d["BMI"] < 18.5).astype(int)
    d["BMI_Normal"]          = ((d["BMI"] >= 18.5) & (d["BMI"] < 25)).astype(int)
    d["BMI_Overweight"]      = ((d["BMI"] >= 25) & (d["BMI"] < 30)).astype(int)
    d["BMI_Obese"]           = (d["BMI"] >= 30).astype(int)
    d["BMI_SevereObese"]     = (d["BMI"] >= 35).astype(int)
    d["BMI_x_PhysActivity"]  = d["BMI"] * d["PhysActivity"]
    d["Age_x_BMI"]           = d["Age"] * d["BMI"]
    d["GenHlth_x_BMI"]       = d["GenHlth"] * d["BMI"]
    d["CardioRisk"]          = (d["HighBP"] * d["HighChol"] +
                                 d["HeartDiseaseorAttack"] + d["Stroke"])
    d["LifestyleScore"]      = (d["PhysActivity"] + d["Fruits"] +
                                 d["Veggies"] - d["Smoker"] -
                                 d["HvyAlcoholConsump"])
    d["HealthAccess"]        = d["AnyHealthcare"] - d["NoDocbcCost"]
    d["HealthBurden"]        = (d["MentHlth"] + d["PhysHlth"]) / 60.0
    d["GenHlth_Sq"]          = d["GenHlth"] ** 2
    d["BMI_Sq"]              = d["BMI"] ** 2
    d["Age_Sq"]              = d["Age"] ** 2
    d["SocioEconomic"]       = d["Income"] * d["Education"]
    d["AgeGroup_Senior"]     = (d["Age"] >= 9).astype(int)
    d["AgeGroup_Middle"]     = ((d["Age"] >= 5) & (d["Age"] < 9)).astype(int)
    d["AgeGroup_Young"]      = (d["Age"] < 5).astype(int)
    d["HighRiskFlag"]        = ((d["HighBP"] == 1) &
                                 (d["HighChol"] == 1) &
                                 (d["BMI_Obese"] == 1)).astype(int)
    d["AlcSmoke"]            = d["HvyAlcoholConsump"] + d["Smoker"]
    d["UncheckedRisk"]       = ((d["CholCheck"] == 0) &
                                 (d["CompositeRiskScore"] > 2)).astype(int)
    d["MentalPhysRatio"]     = (d["MentHlth"] / (d["PhysHlth"] + 1))
    return d
print("✅ engineer_features defined.")

✅ engineer_features defined.


### Load Outputs from Notebooks 2 & 3

In [16]:
TARGET            = "Diabetes_binary"
FEATURES_ORIGINAL = load_pkl(DIRS["reports"] / "features_original.pkl")
binary_cols       = load_pkl(DIRS["reports"] / "binary_cols.pkl")
cont_cols         = load_pkl(DIRS["reports"] / "cont_cols.pkl")
df_preprocessed   = load_pkl(DIRS["reports"] / "df_preprocessed.pkl")
X_train_raw       = load_pkl(DIRS["reports"] / "X_train_scaled.pkl")  # index reference
X_test_raw        = load_pkl(DIRS["reports"] / "X_test_scaled.pkl")
y_train           = load_pkl(DIRS["reports"] / "y_train.pkl")
y_test            = load_pkl(DIRS["reports"] / "y_test.pkl")
baseline_results  = load_pkl(DIRS["reports"] / "baseline_results.pkl")

all_results["01_Baseline"] = baseline_results
print("✅ Previous outputs loaded.")

✅ Previous outputs loaded.


### Step 9 — Feature Engineering

In [17]:
section("STEP 9 — FEATURE ENGINEERING")

df_eng = engineer_features(df_preprocessed.drop(columns=[TARGET]))
df_eng[TARGET] = df_preprocessed[TARGET].values
FEATURES_ENGINEERED = [c for c in df_eng.columns if c != TARGET]
new_feats = [c for c in FEATURES_ENGINEERED if c not in FEATURES_ORIGINAL]

print(f"  Original features  : {len(FEATURES_ORIGINAL)}")
print(f"  New features added : {len(new_feats)}")
print(f"  Total features     : {len(FEATURES_ENGINEERED)}")
print(f"  New features: {new_feats}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 9 — FEATURE ENGINEERING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Original features  : 21
  New features added : 24
  Total features     : 45
  New features: ['CompositeRiskScore', 'BMI_Underweight', 'BMI_Normal', 'BMI_Overweight', 'BMI_Obese', 'BMI_SevereObese', 'BMI_x_PhysActivity', 'Age_x_BMI', 'GenHlth_x_BMI', 'CardioRisk', 'LifestyleScore', 'HealthAccess', 'HealthBurden', 'GenHlth_Sq', 'BMI_Sq', 'Age_Sq', 'SocioEconomic', 'AgeGroup_Senior', 'AgeGroup_Middle', 'AgeGroup_Young', 'HighRiskFlag', 'AlcSmoke', 'UncheckedRisk', 'MentalPhysRatio']


In [18]:
# Split and scale engineered data
X_eng = df_eng.drop(columns=[TARGET])
y_eng = df_eng[TARGET]
X_eng_train = X_eng.loc[X_train_raw.index]
X_eng_test  = X_eng.loc[X_test_raw.index]
y_eng_train = y_eng.loc[y_train.index]
y_eng_test  = y_eng.loc[y_test.index]

scaler_eng = RobustScaler()
X_eng_train_sc = pd.DataFrame(
    scaler_eng.fit_transform(X_eng_train),
    columns=X_eng_train.columns, index=X_eng_train.index)
X_eng_test_sc = pd.DataFrame(
    scaler_eng.transform(X_eng_test),
    columns=X_eng_test.columns, index=X_eng_test.index)

save_pkl(scaler_eng,          DIRS["scalers"]  / "robust_scaler_eng.pkl", "Scaler(eng)")
save_pkl(X_eng_train_sc,      DIRS["reports"]  / "X_eng_train_sc.pkl",    "X_eng_train")
save_pkl(X_eng_test_sc,       DIRS["reports"]  / "X_eng_test_sc.pkl",     "X_eng_test")
save_pkl(y_eng_train,         DIRS["reports"]  / "y_eng_train.pkl",       "y_eng_train")
save_pkl(y_eng_test,          DIRS["reports"]  / "y_eng_test.pkl",        "y_eng_test")
save_pkl(FEATURES_ENGINEERED, DIRS["features"] / "features_eng.pkl",      "Eng features")
print("✅ Engineered features scaled and saved.")

  [SAVED] Scaler(eng) → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\scalers\robust_scaler_eng.pkl
  [SAVED] X_eng_train → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\X_eng_train_sc.pkl
  [SAVED] X_eng_test → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\X_eng_test_sc.pkl
  [SAVED] y_eng_train → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\y_eng_train.pkl
  [SAVED] y_eng_test → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\y_eng_test.pkl
  [SAVED] Eng features → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\feature_lists\features_eng.pkl
✅ Engineered features scaled and saved.


In [19]:
# New feature distributions by class
ncols_n = 4
nrows_n = (len(new_feats) + ncols_n - 1) // ncols_n
fig, axes = plt.subplots(nrows_n, ncols_n,
                          figsize=(ncols_n * 4, nrows_n * 3.5))
axes = axes.flatten()
for i, col in enumerate(new_feats):
    for cls, color, label in [(0, COLOR_NEG, "No Diabetes"),
                               (1, COLOR_POS, "Diabetes")]:
        sub = df_eng[df_eng[TARGET] == cls][col]
        axes[i].hist(sub, bins=25, alpha=0.6, color=color,
                     label=label, density=True, edgecolor="white")
    axes[i].set_title(col, fontweight="bold", fontsize=8)
    axes[i].legend(fontsize=6)
for j in range(len(new_feats), len(axes)): axes[j].set_visible(False)
fig.suptitle("New Engineered Features — Distribution by Class",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_eng"] / "01_eng_distributions.png")
plt.show()

  [PLOT]  01_eng_distributions.png


### Step 10 — Models on Engineered Features (Phase 2)

In [20]:
section("STEP 10 — MODELS ON ENGINEERED FEATURES")

eng_results, eng_proba, eng_trained = {}, {}, {}
for name, model in get_models().items():
    metrics = evaluate_model(
        model, X_eng_train_sc, y_eng_train,
        X_eng_test_sc, y_eng_test, name,
        cv_obj=cv, phase="02_Engineered")
    eng_results[name] = metrics
    eng_trained[name] = model
    if hasattr(model, "predict_proba"):
        eng_proba[name] = model.predict_proba(X_eng_test_sc)[:, 1]

all_results["02_Engineered"] = eng_results
for name, model in eng_trained.items():
    save_pkl(model, DIRS["models"] / f"eng_{name.replace(' ','_')}.pkl", name)
save_pkl(eng_results, DIRS["reports"] / "eng_results.pkl", "Eng results")
save_pkl(eng_trained, DIRS["reports"] / "eng_trained.pkl", "Eng trained")
save_pkl(eng_proba,   DIRS["reports"] / "eng_proba.pkl",   "Eng proba")
print("\n✅ Engineered models trained and saved.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 10 — MODELS ON ENGINEERED FEATURES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Logistic Regression                 Acc=0.7153  F1=0.4541  Recall=0.7742  Prec=0.3212  AUC=0.8160  t=7.1s

────────────────────────────────────────────────────────────
  📊 Classification Report — Logistic Regression
     Phase: 02_Engineered
────────────────────────────────────────────────────────────
                 precision    recall  f1-score   support

No Diabetes (0)     0.9453    0.7047    0.8074     38876
   Diabetes (1)     0.3212    0.7742    0.4541      7019

       accuracy                         0.7153     45895
      macro avg     0.6333    0.7394    0.6307     45895
   weighted avg     0.8499    0.7153    0.7534     45895

────────────────────────────────────────────────────────────
  Random Forest                       Acc=0.8423  F1=0.2253  Recall=0.1499  Prec=0.4533  AUC=0.7766  t=2

In [21]:
# Original vs Engineered comparison
common = [k for k in baseline_results if k in eng_results]
metrics_to_cmp = ["precision","recall","f1","roc_auc","avg_precision"]
fig, axes = plt.subplots(1, len(metrics_to_cmp), figsize=(20, 5))
for ax_i, metric in enumerate(metrics_to_cmp):
    x_c = np.arange(len(common)); ww = 0.35
    axes[ax_i].bar(x_c - ww/2,
                   [baseline_results[k][metric] for k in common],
                   width=ww, color=COLOR_NEG, label="Original", edgecolor="white")
    axes[ax_i].bar(x_c + ww/2,
                   [eng_results[k][metric] for k in common],
                   width=ww, color=COLOR_POS, label="Engineered", edgecolor="white")
    axes[ax_i].set_xticks(x_c)
    axes[ax_i].set_xticklabels(common, rotation=35, ha="right", fontsize=7)
    axes[ax_i].set_title(metric.replace("_"," ").title(), fontweight="bold")
    axes[ax_i].legend(fontsize=8); axes[ax_i].set_ylim(0, 1.0)
fig.suptitle("Original vs Engineered Features — Key Metrics",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_eng_m"] / "01_orig_vs_eng.png")
plt.show()

  [PLOT]  01_orig_vs_eng.png


In [22]:
plot_roc_pr(y_eng_test, eng_proba, "Engineered Models", DIRS["plots_eng_m"])

  [PLOT]  Engineered_Models_roc_pr.png


### Step 11 — Feature Importance Analysis

In [23]:
section("STEP 11 — FEATURE IMPORTANCE ANALYSIS")

# Random Forest MDI
rf_m   = eng_trained.get("Random Forest", eng_trained.get("Extra Trees"))
rf_imp = pd.Series(rf_m.feature_importances_,
                   index=X_eng_train_sc.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 11))
colors_fi = [COLOR_POS if i < TOP_N else "#CCCCCC" for i in range(len(rf_imp))]
ax.barh(rf_imp.index[::-1], rf_imp.values[::-1],
        color=colors_fi[::-1], edgecolor="white")
ax.set_title("Random Forest — Feature Importances (All Features)",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Mean Decrease in Impurity (MDI)")
fig.tight_layout()
savefig(fig, DIRS["plots_fi"] / "01_rf_importance.png")
plt.show()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 11 — FEATURE IMPORTANCE ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [PLOT]  01_rf_importance.png


In [24]:
# Mutual Information
mi_scores = mutual_info_classif(X_eng_train_sc, y_eng_train, random_state=SEED)
mi_series = pd.Series(mi_scores,
                       index=X_eng_train_sc.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(mi_series.index[::-1], mi_series.values[::-1],
        color=COLOR_AUC, edgecolor="white")
ax.set_title("Mutual Information — Feature Relevance to Target",
             fontweight="bold", fontsize=13)
ax.set_xlabel("MI Score")
fig.tight_layout()
savefig(fig, DIRS["plots_fi"] / "03_mutual_information.png")
plt.show()

  [PLOT]  03_mutual_information.png


In [25]:
# Combined Rank → Top-N Selection
rank_rf  = rf_imp.rank(ascending=False)
rank_mi  = mi_series.rank(ascending=False)
combined = (rank_rf + rank_mi).sort_values()
top_features = list(combined.index[:TOP_N])

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(combined.index[::-1][:TOP_N * 2],
        combined.values[::-1][:TOP_N * 2],
        color=PALETTE[1], edgecolor="white")
ax.set_title(f"Combined Rank (RF MDI + MI) — Top {TOP_N * 2} Features",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Combined Rank Score (lower = more important)")
ax.invert_xaxis()
fig.tight_layout()
savefig(fig, DIRS["plots_fi"] / "04_combined_rank.png")
plt.show()

save_pkl(rf_imp,       DIRS["features"] / "rf_importance.pkl",  "RF importance")
save_pkl(mi_series,    DIRS["features"] / "mi_scores.pkl",      "MI scores")
save_pkl(combined,     DIRS["features"] / "combined_rank.pkl",  "Combined rank")
save_pkl(top_features, DIRS["features"] / "top_features.pkl",   "Top features")

print(f"\n  ✅ Top {TOP_N} features selected:")
for i, f in enumerate(top_features, 1):
    print(f"    {i:>2}. {f}")

  [PLOT]  04_combined_rank.png
  [SAVED] RF importance → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\feature_lists\rf_importance.pkl
  [SAVED] MI scores → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\feature_lists\mi_scores.pkl
  [SAVED] Combined rank → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\feature_lists\combined_rank.pkl
  [SAVED] Top features → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\feature_lists\top_features.pkl

  ✅ Top 15 features selected:
     1. GenHlth_x_BMI
     2. Age_x_BMI
     3. CompositeRiskScore
     4. CardioRisk
     5. GenHlth_Sq
     6. HighBP
     7. BMI_Sq
     8. BMI
     9. GenHlth
    10. Age_Sq
    11. Income
    12. Age
    13. SocioEconomic
    14. Education
    15. HighChol


### Step 12 — Models on Selected Top Features (Phase 3)

In [26]:
section("STEP 12 — MODELS ON SELECTED TOP FEATURES")

X_sel_train = X_eng_train_sc[top_features]
X_sel_test  = X_eng_test_sc[top_features]

sel_results, sel_proba, sel_trained = {}, {}, {}
for name, model in get_models().items():
    metrics = evaluate_model(
        model, X_sel_train, y_eng_train,
        X_sel_test, y_eng_test, name,
        cv_obj=cv, phase="03_Selected")
    sel_results[name] = metrics
    sel_trained[name] = model
    if hasattr(model, "predict_proba"):
        sel_proba[name] = model.predict_proba(X_sel_test)[:, 1]

all_results["03_Selected"] = sel_results
for name, model in sel_trained.items():
    save_pkl(model, DIRS["models"] / f"sel_{name.replace(' ','_')}.pkl", name)
save_pkl(sel_results,  DIRS["reports"] / "sel_results.pkl",  "Selected results")
save_pkl(sel_trained,  DIRS["reports"] / "sel_trained.pkl",  "Selected trained")
save_pkl(sel_proba,    DIRS["reports"] / "sel_proba.pkl",    "Selected proba")
save_pkl(all_results,  DIRS["reports"] / "all_results.pkl",  "All results")
print("\n✅ Phase 3 complete. All results saved.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 12 — MODELS ON SELECTED TOP FEATURES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Logistic Regression                 Acc=0.7118  F1=0.4480  Recall=0.7646  Prec=0.3168  AUC=0.8108  t=1.2s

────────────────────────────────────────────────────────────
  📊 Classification Report — Logistic Regression
     Phase: 03_Selected
────────────────────────────────────────────────────────────
                 precision    recall  f1-score   support

No Diabetes (0)     0.9429    0.7023    0.8050     38876
   Diabetes (1)     0.3168    0.7646    0.4480      7019

       accuracy                         0.7118     45895
      macro avg     0.6299    0.7335    0.6265     45895
   weighted avg     0.8472    0.7118    0.7504     45895

────────────────────────────────────────────────────────────
  Random Forest                       Acc=0.7977  F1=0.3030  Recall=0.2875  Prec=0.3202  AUC=0.7242  t=1

In [27]:
# Three-Phase Comparison
common3 = [k for k in baseline_results if k in eng_results and k in sel_results]
phase_metrics = ["precision","recall","f1","roc_auc"]
fig, axes = plt.subplots(1, len(phase_metrics), figsize=(22, 5))
for ax_i, metric in enumerate(phase_metrics):
    x_c = np.arange(len(common3)); ww = 0.25
    axes[ax_i].bar(x_c - ww, [baseline_results[k][metric] for k in common3],
                   width=ww, color=COLOR_NEG, label="Original", edgecolor="white")
    axes[ax_i].bar(x_c,      [eng_results[k][metric] for k in common3],
                   width=ww, color=COLOR_POS, label="Engineered", edgecolor="white")
    axes[ax_i].bar(x_c + ww, [sel_results[k][metric] for k in common3],
                   width=ww, color=COLOR_ACC, label="Selected", edgecolor="white")
    axes[ax_i].set_xticks(x_c)
    axes[ax_i].set_xticklabels(common3, rotation=35, ha="right", fontsize=7)
    axes[ax_i].set_title(metric.replace("_"," ").title(), fontweight="bold")
    axes[ax_i].legend(fontsize=8); axes[ax_i].set_ylim(0, 1.05)
fig.suptitle("Three-Phase Comparison — Original → Engineered → Selected",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_sel"] / "01_three_phase_comparison.png")
plt.show()

  [PLOT]  01_three_phase_comparison.png


### ✅ Notebook 4 Complete
All phase results saved. Run Notebook 5 next.